# 01_07_acq_dash

Отдельная тетрадка для обновления дашборда по эквайрингу за период Jan-Jun 2026.

Что делает тетрадка:
- считает `final_df` помесячно (Jan-Jun 2026) из Озера;
- собирает единый `final_df_period_df`;
- сравнивает Озеро vs Excel для Jan-Apr 2026 по 9 метрикам;
- помечает May-Jun как `no_excel_reference`;
- сохраняет CSV с данными периода и Excel со сравнением (`monthly_compare`, `period_total`, `reference_coverage`).

In [ ]:
import re
import threading
import time
from decimal import Decimal, InvalidOperation
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def safe_divergence_pct(delta, reference):
    if pd.isna(reference) or reference == 0:
        return np.nan
    return abs(delta) / abs(reference) * 100.0

In [ ]:
# Конфиг периода и путей
period_start = '2026-01-01'
period_end = '2026-06-01'
excel_header = 0
run_invalidate_metadata = False

period_months = pd.date_range(period_start, period_end, freq='MS')

excel_reference_by_month = {
    '2026-01': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
    '2026-02': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
    '2026-03': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
    '2026-04': '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx',
}

output_dir = Path('/home/jovyan/documents/Equaring/Data')
output_csv_path = output_dir / 'final_df_period_2026_01_2026_06.csv'
output_compare_excel_path = output_dir / 'final_df_compare_2026_01_2026_06.xlsx'

print('Период расчета:')
print([d.strftime('%Y-%m') for d in period_months])
print('Excel-референсы Jan-Apr:')
for k, v in excel_reference_by_month.items():
    print(f'  {k}: {v}')
print('May-Jun: no_excel_reference')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

invalidate_tables = [
    'ods_alpha.scd1_agreements', 'ods_alpha.scd1_companies', 'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext', 'cdiul.ext_id_org', 'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals', 'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_acq', 'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids', 'ods.scd1_z_r2_ip_merchants', 'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix', 'ods.scd1_z_cl_corp', 'ods.scd1_z_depart',
    'ods.scd1_z_branch', 'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    with imp:
        for t in invalidate_tables:
            imp.execute(f'invalidate metadata {t}')
    print('Invalidate metadata completed')
else:
    print('Invalidate metadata skipped')

In [ ]:
def compute_final_df_for_month(report_month, imp):
    report_month_ts = pd.to_datetime(report_month)
    month_start = report_month_ts.strftime('%Y-%m-%d')
    month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    report_month_label = report_month_ts.strftime('%Y-%m')
    snapshot_month_start = month_start

    month_total_start_ts = time.perf_counter()

    def _log_progress(message):
        print(f"[{report_month_label}][{time.strftime('%H:%M:%S')}] {message}", flush=True)

    def _run_impala_fetch(step_name, sql_text, mem_limit='8g', heartbeat_sec=60):
        step_start_ts = time.perf_counter()
        _log_progress(f"{step_name}: start")

        heartbeat_stop = threading.Event()

        def _heartbeat():
            while not heartbeat_stop.wait(heartbeat_sec):
                running_sec = round(time.perf_counter() - step_start_ts, 1)
                _log_progress(f"{step_name}: still running ({running_sec}s)")

        heartbeat_thread = threading.Thread(target=_heartbeat, daemon=True)
        heartbeat_thread.start()

        try:
            with imp:
                imp.execute(f'set MEM_LIMIT={mem_limit}')
                df_out = imp.fetch(sql_text)
        except Exception:
            elapsed_sec = round(time.perf_counter() - step_start_ts, 2)
            _log_progress(f"{step_name}: failed after {elapsed_sec}s")
            raise
        finally:
            heartbeat_stop.set()
            heartbeat_thread.join(timeout=0.2)

        elapsed_sec = round(time.perf_counter() - step_start_ts, 2)
        rows = len(df_out) if isinstance(df_out, pd.DataFrame) else 0
        _log_progress(f"{step_name}: done ({elapsed_sec}s, rows={rows:,})")
        return df_out

    _log_progress('start расчет final_df')

    # 01_sa_perimeter
    sql_sa_perimeter = f"""
    select distinct
      cast(a.n_agr as string) as n_agr,
      cast(a.abs_agr_id as string) as agr_id,
      cast(a.n_cmp_client as string) as n_cmp_client,
      cast(a.c_agr_number as string) as contract_number,
      cast(a.d_valid_from as date) as d_valid_from,
      cast(a.d_valid_to as date) as d_valid_to,
      regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
      cast(c.c_cmp_name as string) as company_name
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where upper(trim(cast(a.acq_class as string))) = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
      and c.c_inn is not null
      and exists (
          select 1
          from ods_alpha.scd1_agr_terms t
          where cast(t.n_agr as string) = cast(a.n_agr as string)
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
      )
    """

    sa_df = _run_impala_fetch('01_sa_perimeter', sql_sa_perimeter, mem_limit='8g')

    if sa_df is None:
        sa_df = pd.DataFrame()
    if not sa_df.empty:
        sa_df['inn'] = sa_df['inn'].map(normalize_inn)
        sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

    # 02_cdi_map
    inn_values = sorted([
        x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
    ])
    inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

    if not inn_values:
        cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
    else:
        sql_cdi = f"""
        with ocrm_current as (
          select
            regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
            cast(soe.row_id as string) as row_id,
            trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
            trim(cast(soe.x_area_resp as string)) as ssp_ocrm,
            row_number() over (
              partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
              order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
            ) as rn
          from ocrm_ul.s_org_ext soe
          where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
            and coalesce(soe.x_removed_flg, 'N') = 'N'
            and coalesce(soe.x_duplicate_flg, 'N') = 'N'
        ),
        ocrm_one as (
          select inn, row_id, ssp_ocrm, x_area_resp_norm
          from ocrm_current
          where rn = 1
        )
        select
          o.inn,
          o.ssp_ocrm,
          o.x_area_resp_norm,
          cast(e.party_id as string) as cdi_id
        from ocrm_one o
        left join cdiul.ext_id_org e
          on cast(e.cmo_ext_party_source_id as string) = o.row_id
         and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
        """

        cdi_map_df = _run_impala_fetch('02_cdi_map', sql_cdi, mem_limit='8g')

    if cdi_map_df is None:
        cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
    if not cdi_map_df.empty:
        cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
        cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
        cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

    # 03_cft_map
    cdi_values = sorted([
        x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
    ])
    cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

    if not cdi_values:
        cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
    else:
        sql_cft = f"""
        select
          cast(e.party_id as string) as cdi_id,
          cast(e.cmo_ext_party_source_id as string) as cft_id
        from cdiul.ext_id_org e
        where cast(e.party_id as string) in ({cdi_sql_list})
          and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
        """

        cft_map_df = _run_impala_fetch('03_cft_map', sql_cft, mem_limit='8g')

    if cft_map_df is None:
        cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
    if not cft_map_df.empty:
        cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
        cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
        cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

    # 04_operational_metrics (hybrid_all optimized: 547-like core + fallback only for missing n_agr)
    if sa_df.empty:
        cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
    else:
        n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
        agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

        def _build_sql_cmp_hybrid(use_m_acq_filter=True):
            m_acq_filter_sql = "and coalesce(upper(trim(cast(m.acq_class as string))), 'NA') <> 'SV'" if use_m_acq_filter else ''

            return f"""
            with sa_agr as (
              select distinct
                cast(a.n_agr as string) as n_agr,
                cast(a.n_cmp_client as string) as n_cmp_client
              from ods_alpha.scd1_agreements a
              where cast(a.n_agr as string) in ({agr_in})
            ),

            terms_547 as (
              select distinct
                sa.n_agr,
                sa.n_cmp_client,
                cast(t.c_nmrc as string) as c_nmrc
              from sa_agr sa
              join ods_alpha.scd1_agr_terms t
                on cast(t.n_agr as string) = sa.n_agr
              join ods_alpha.scd1_merchants m
                on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
              where t.c_nmrc is not null
                and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
                {m_acq_filter_sql}
                and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
                and coalesce(t.ods_deleted_flg, '0') <> '1'
                and coalesce(m.ods_deleted_flg, '0') <> '1'
            ),
            pos_period as (
              select distinct
                cast(p.c_nmrc as string) as c_nmrc,
                cast(p.c_pos_serial as string) as c_pos_serial,
                cast(p.c_nter as string) as c_nter
              from ods_alpha.scd1_pos_terminals p
              where p.c_pos_serial is not null
                and cast(p.d_ter_install as date) is not null
                and cast(p.d_ter_install as date) < date_add(cast('{month_end}' as date), 1)
                and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
                and coalesce(p.ods_deleted_flg, '0') <> '1'
            ),
            term_active_547 as (
              select
                t.n_agr,
                t.n_cmp_client,
                t.c_nmrc,
                p.c_pos_serial,
                p.c_nter
              from terms_547 t
              left join pos_period p
                on p.c_nmrc = t.c_nmrc
            ),
            retl_547 as (
              select n_agr, count(distinct c_nmrc) as retl_cnt_547
              from term_active_547
              group by n_agr
            ),
            term_547 as (
              select n_agr, count(distinct c_pos_serial) as term_cnt_547
              from term_active_547
              group by n_agr
            ),
            amort_547 as (
              select x.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization_547
              from (
                select distinct n_agr, c_nter
                from term_active_547
                where c_nter is not null
              ) x
              left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
                on cast(am.c_nter as string) = x.c_nter
              group by x.n_agr
            ),

            fallback_agrs as (
              select sa.n_agr, sa.n_cmp_client
              from sa_agr sa
              left join retl_547 r547 on r547.n_agr = sa.n_agr
              left join term_547 t547 on t547.n_agr = sa.n_agr
              where r547.n_agr is null and t547.n_agr is null
            ),
            old_nmrc as (
              select fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
              from fallback_agrs fa
              join ods_alpha.scd1_merchants mm
                on cast(mm.n_cmp as string) = fa.n_cmp_client
              where mm.c_nmrc is not null
                and coalesce(mm.ods_deleted_flg, '0') <> '1'
              group by fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string)
            ),
            old_term_active as (
              select
                o.n_agr,
                o.n_cmp_client,
                o.c_nmrc,
                cast(t.c_nter as string) as c_nter
              from old_nmrc o
              join ods_alpha.scd1_pos_terminals t
                on cast(t.c_nmrc as string) = o.c_nmrc
              where t.c_nter is not null
                and coalesce(t.ods_deleted_flg, '0') <> '1'
                and cast(t.d_ter_install as date) is not null
                and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
                and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
              group by o.n_agr, o.n_cmp_client, o.c_nmrc, cast(t.c_nter as string)
            ),
            old_retl as (
              select n_agr, count(distinct c_nmrc) as retl_cnt_old
              from old_term_active
              group by n_agr
            ),
            old_term as (
              select n_agr, count(distinct c_nter) as term_cnt_old
              from old_term_active
              group by n_agr
            ),
            old_amort as (
              select x.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization_old
              from (
                select distinct n_agr, c_nter
                from old_term_active
                where c_nter is not null
              ) x
              left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
                on cast(am.c_nter as string) = x.c_nter
              group by x.n_agr
            )

            select
              sa.n_agr,
              sa.n_cmp_client,
              coalesce(r547.retl_cnt_547, rold.retl_cnt_old) as retl_cnt,
              coalesce(t547.term_cnt_547, told.term_cnt_old) as term_cnt,
              coalesce(a547.amortization_547, aold.amortization_old, 0.0) as amortization,
              case
                when r547.n_agr is not null or t547.n_agr is not null then '547_like'
                when rold.n_agr is not null or told.n_agr is not null then 'fallback_old'
                else 'no_data'
              end as term_calc_source
            from sa_agr sa
            left join retl_547 r547 on r547.n_agr = sa.n_agr
            left join term_547 t547 on t547.n_agr = sa.n_agr
            left join amort_547 a547 on a547.n_agr = sa.n_agr
            left join old_retl rold on rold.n_agr = sa.n_agr
            left join old_term told on told.n_agr = sa.n_agr
            left join old_amort aold on aold.n_agr = sa.n_agr
            """

        used_m_acq_filter = True
        try:
            cmp_df = _run_impala_fetch('04_operational_metrics_hybrid', _build_sql_cmp_hybrid(use_m_acq_filter=True), mem_limit='16g')
        except Exception as exc_hybrid:
            used_m_acq_filter = False
            _log_progress(f"04_operational_metrics_hybrid: m.acq_class filter disabled due to {type(exc_hybrid).__name__}")
            cmp_df = _run_impala_fetch('04_operational_metrics_hybrid_fallback', _build_sql_cmp_hybrid(use_m_acq_filter=False), mem_limit='16g')

    if cmp_df is None:
        cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
    if not cmp_df.empty:
        cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
        cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)
        cmp_df['retl_cnt'] = pd.to_numeric(cmp_df['retl_cnt'], errors='coerce')
        cmp_df['term_cnt'] = pd.to_numeric(cmp_df['term_cnt'], errors='coerce')
        cmp_df['amortization'] = pd.to_numeric(cmp_df['amortization'], errors='coerce')

    # 05_transaction_metrics
    if sa_df.empty:
        trx_df = pd.DataFrame(columns=['n_agr', 'inn', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt', 'active_terms'])
    else:
        n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
        agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

        sql_trx = f"""
        with sa_agr as (
          select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.n_cmp_client as string) as n_cmp_client,
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
          from ods_alpha.scd1_agreements a
          left join ods_alpha.scd1_companies c
            on c.n_cmp = a.n_cmp_client
          where cast(a.n_agr as string) in ({agr_in})
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
        ),
        trx_base_raw as (
          select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
          from ods_alpha.scd1_trx t
          left join ods_alpha.scd1_base24_fiids fa
            on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
          where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
            and t.c_nter is not null
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and t.c_trx_class = 'SA'
            and t.c_trx_type = 'S01'
            and coalesce(t.cf_trx_stat, '') <> 'R'
            and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
        ),
        trx_base as (
          select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
          from trx_base_raw
          group by n_trx
        ),
        ta_raw as (
          select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
          from ods_alpha.scd1_trx_acq a
          join trx_base tb on tb.n_trx = cast(a.n_trx as string)
          where cast(a.n_agr as string) in ({agr_in})
        ),
        ta as (
          select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
          from ta_raw
          group by n_trx, n_agr
        ),
        trx_keys as (
          select distinct n_trx
          from ta
        ),
        trx_int_agg as (
          select cast(i.n_trx as string) as n_trx, sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
          from ods_alpha.scd1_trx_int i
          join trx_keys k on k.n_trx = cast(i.n_trx as string)
          group by cast(i.n_trx as string)
        ),
        tj as (
          select ta.n_agr, sa.n_cmp_client, sa.inn, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
          from ta
          join trx_base tb on tb.n_trx = ta.n_trx
          left join sa_agr sa on sa.n_agr = ta.n_agr
        ),
        trx_agg as (
          select
            tj.n_agr,
            count(distinct tj.n_trx) as trx_cnt,
            sum(tj.n_amt_src) as trx_sum,
            sum(tj.n_amt_tax) as commission_from_ops,
            sum(coalesce(i.n_amt_fee, 0.0)) as int_component
          from tj
          left join trx_int_agg i on i.n_trx = tj.n_trx
          group by tj.n_agr
        ),
        active_term_agg_ncmp as (
          select
            tj.n_cmp_client,
            count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
          from tj
          where tj.n_cmp_client is not null
          group by tj.n_cmp_client
        ),
        active_term_agg_agr as (
          select
            tj.n_agr,
            count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_terms
          from tj
          where tj.n_agr is not null
          group by tj.n_agr
        )
        select
          t.n_agr,
          sa.inn,
          t.trx_cnt,
          t.trx_sum,
          t.commission_from_ops,
          t.int_component,
          sa.n_cmp_client,
          a.active_term_cnt,
          aa.active_terms
        from trx_agg t
        left join sa_agr sa on sa.n_agr = t.n_agr
        left join active_term_agg_ncmp a on a.n_cmp_client = sa.n_cmp_client
        left join active_term_agg_agr aa on aa.n_agr = t.n_agr
        """

        trx_df = _run_impala_fetch('05_transaction_metrics', sql_trx, mem_limit='16g')

    if trx_df is None:
        trx_df = pd.DataFrame(columns=['n_agr', 'inn', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt', 'active_terms'])
    if not trx_df.empty:
        trx_df['n_agr'] = trx_df['n_agr'].astype(str)
        trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)
        trx_df['inn'] = trx_df['inn'].map(normalize_inn)
        trx_df['active_term_cnt'] = pd.to_numeric(trx_df['active_term_cnt'], errors='coerce')
        trx_df['active_terms'] = pd.to_numeric(trx_df['active_terms'], errors='coerce')

    active_term_df = (
        trx_df[['n_cmp_client', 'active_term_cnt']]
        .dropna(subset=['n_cmp_client'])
        .drop_duplicates(subset=['n_cmp_client'])
        if len(trx_df)
        else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
    )

    # 06_r2_legacy_attrs
    cft_values = sorted([
        x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
    ])
    cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

    if not cft_values:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])
    else:
        sql_r2 = f"""
        with r2 as (
          select
            cast(m.id as string) as r2_id,
            cast(m.c_cl_org as string) as cft_id,
            cast(m.c_depart as string) as c_depart,
            cast(m.c_tariff_plan as string) as c_tariff_plan
          from ods.scd1_z_r2_ip_merchants m
          where cast(m.c_cl_org as string) in ({cft_sql_list})
        ),
        fix as (
          select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
          from ods.scd1_z_r2_tariff_tune tt
          left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
          group by cast(tt.c_tariff_plan as string)
        )
        select
          r2.r2_id,
          r2.cft_id,
          cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
          cast(dep.c_name as string) as vsp_name,
          cast(dep.c_num as string) as vsp_code,
          cast(br.c_shortlabel as string) as filial_rf,
          cast(tp.c_name as string) as tariff_name_legacy,
          cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly_legacy
        from r2
        left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
        left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
        left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
        left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
        left join fix on fix.c_tariff_plan = r2.c_tariff_plan
        """

        r2_df = _run_impala_fetch('06_r2_legacy_attrs', sql_r2, mem_limit='8g')

    if r2_df is None:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])

    if not r2_df.empty:
        for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy']:
            r2_df[c] = r2_df[c].astype(str)
        r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

    # 07_base_merge
    base_df = sa_df.copy()

    if not cdi_map_df.empty and not base_df.empty:
        base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
    else:
        base_df['cdi_id'] = None
        base_df['ssp_ocrm'] = None

    if not cft_map_df.empty and not base_df.empty:
        base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
    else:
        base_df['cft_id'] = None

    if not r2_df.empty and not base_df.empty:
        base_df = base_df.merge(r2_df, on='cft_id', how='left')
    else:
        for col in ['r2_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy']:
            base_df[col] = None

    if not cmp_df.empty and not base_df.empty:
        base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
    else:
        for col in ['retl_cnt', 'term_cnt', 'amortization']:
            base_df[col] = None

    if not active_term_df.empty and not base_df.empty:
        base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
    else:
        base_df['active_term_cnt'] = None

    if not trx_df.empty and not base_df.empty:
        base_df = base_df.merge(
            trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'active_terms']],
            on='n_agr',
            how='left'
        )
    else:
        for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'active_terms']:
            base_df[col] = None

    # 08_agr_fallback
    if 'agr_id' not in base_df.columns:
        base_df['agr_id'] = None
    if 'r2_id' not in base_df.columns:
        base_df['r2_id'] = None

    agr_before_mask = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
    base_df['agr_id_source'] = 'sa'
    fallback_mask = (~agr_before_mask) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
    base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
    base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

    # 08b_tariff_fix_map (optimized: only tariffs from current base_df)
    section08b_start_ts = time.perf_counter()

    agr_id_scope = (
        base_df.get('agr_id', pd.Series(dtype=object))
        .map(normalize_agr_q1)
        .dropna()
        .astype(str)
        .str.strip()
    )
    agr_id_scope = sorted([x for x in agr_id_scope.unique().tolist() if x])

    if agr_id_scope:
        agr_in = ', '.join([f"'{x}'" for x in agr_id_scope])

        sql_tariff_fix_map = f"""
        with plan_scope as (
          select distinct cast(m.c_tariff_plan as string) as c_tariff_plan
          from ods.scd1_z_r2_ip_merchants m
          where m.c_tariff_plan is not null
            and cast(m.id as string) in ({agr_in})
        )
        select
          ps.c_tariff_plan,
          max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
        from plan_scope ps
        left join ods.scd1_z_r2_tariff_tune tt
          on cast(tt.c_tariff_plan as string) = ps.c_tariff_plan
        left join ods.scd1_z_r2_tariff_fix tf
          on tt.c_tariff = tf.id
        group by ps.c_tariff_plan
        """

        try:
            tariff_fix_map_df = _run_impala_fetch('08b_tariff_fix_map_scoped', sql_tariff_fix_map, mem_limit='8g')
        except Exception as exc_08b:
            _log_progress(f"08b_tariff_fix_map_scoped: failed ({type(exc_08b).__name__}); fallback to full map")
            sql_tariff_fix_map_full = """
            select
              cast(tt.c_tariff_plan as string) as c_tariff_plan,
              max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
            from ods.scd1_z_r2_tariff_tune tt
            left join ods.scd1_z_r2_tariff_fix tf
              on tt.c_tariff = tf.id
            group by cast(tt.c_tariff_plan as string)
            """
            tariff_fix_map_df = _run_impala_fetch('08b_tariff_fix_map_full_fallback', sql_tariff_fix_map_full, mem_limit='8g')
    else:
        tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

    if tariff_fix_map_df is None:
        tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
    if not tariff_fix_map_df.empty:
        tariff_fix_map_df['c_tariff_plan'] = tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip()

    section08b_elapsed_sec = round(time.perf_counter() - section08b_start_ts, 2)
    _log_progress(f"08b summary: agr_id_scope={len(agr_id_scope):,}, tariff_fix_map_rows={len(tariff_fix_map_df):,}, elapsed_sec={section08b_elapsed_sec}")

    # 09_actual_tariff_by_agr
    base_df['agr_id_key'] = base_df['agr_id'].map(normalize_agr_q1)
    base_agr_keys_df = (
        base_df[['agr_id_key']]
        .dropna()
        .drop_duplicates()
        .rename(columns={'agr_id_key': 'agr_id'})
    )

    if len(base_agr_keys_df):
        sql_actual_tariff_by_agr = f"""
        with agr_keys as (
          select distinct cast(a.abs_agr_id as string) as agr_id
          from ods_alpha.scd1_agreements a
          where upper(trim(cast(a.acq_class as string))) = 'SA'
            and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and exists (
              select 1
              from ods_alpha.scd1_agr_terms t
              where cast(t.n_agr as string) = cast(a.n_agr as string)
                and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                and coalesce(t.ods_deleted_flg, '0') <> '1'
            )
        ),
        agr_actual as (
          select
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.n_agr as string) as n_agr_actual,
            cast(a.c_agr_number as string) as contract_number_acq,
            cast(a.d_valid_from as date) as d_valid_from_actual,
            cast(a.d_valid_to as date) as d_valid_to_actual,
            row_number() over (
              partition by cast(a.abs_agr_id as string)
              order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
            ) as rn
          from ods_alpha.scd1_agreements a
          join agr_keys k
            on k.agr_id = cast(a.abs_agr_id as string)
          where cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
        ),
        merchant_one as (
          select
            cast(m.id as string) as agr_id,
            cast(m.c_tariff_plan as string) as c_tariff_plan,
            row_number() over (
              partition by cast(m.id as string)
              order by cast(m.c_tariff_plan as string) desc
            ) as rn
          from ods.scd1_z_r2_ip_merchants m
          join agr_keys k
            on k.agr_id = cast(m.id as string)
        )
        select
          m.agr_id,
          aa.n_agr_actual,
          aa.contract_number_acq,
          aa.d_valid_from_actual,
          aa.d_valid_to_actual,
          m.c_tariff_plan,
          cast(tp.c_name as string) as tariff_name_actual
        from merchant_one m
        left join agr_actual aa
          on aa.agr_id = m.agr_id
         and aa.rn = 1
        left join ods.scd1_z_r2_tariff_plan tp
          on cast(tp.id as string) = m.c_tariff_plan
        where m.rn = 1
        """

        actual_tariff_df = _run_impala_fetch('09_actual_tariff_by_agr', sql_actual_tariff_by_agr, mem_limit='12g')
    else:
        actual_tariff_df = pd.DataFrame(columns=[
            'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
            'c_tariff_plan', 'tariff_name_actual'
        ])

    if actual_tariff_df is None:
        actual_tariff_df = pd.DataFrame(columns=[
            'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
            'c_tariff_plan', 'tariff_name_actual'
        ])

    if not actual_tariff_df.empty:
        for c in ['agr_id', 'n_agr_actual', 'contract_number_acq', 'c_tariff_plan', 'tariff_name_actual']:
            actual_tariff_df[c] = actual_tariff_df[c].astype(str).str.strip()
        actual_tariff_df = actual_tariff_df.merge(base_agr_keys_df, on='agr_id', how='inner')

    actual_tariff_df = actual_tariff_df.rename(columns={'agr_id': 'agr_id_key'})

    if 'c_tariff_plan' not in actual_tariff_df.columns:
        actual_tariff_df['c_tariff_plan'] = None

    if not actual_tariff_df.empty and not tariff_fix_map_df.empty:
        actual_tariff_df = actual_tariff_df.merge(tariff_fix_map_df, on='c_tariff_plan', how='left')
    else:
        actual_tariff_df['commission_monthly_fix'] = None

    actual_tariff_df['commission_monthly_actual'] = actual_tariff_df.get('commission_monthly_fix')
    if 'commission_monthly_fix' in actual_tariff_df.columns:
        actual_tariff_df = actual_tariff_df.drop(columns=['commission_monthly_fix'])

    # 10_apply_tariff_fix_and_formulas -> final_df
    required_tariff_cols = [
        'agr_id_key', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual',
        'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
    ]
    for col in required_tariff_cols:
        if col not in actual_tariff_df.columns:
            actual_tariff_df[col] = None

    merge_payload_cols = [c for c in required_tariff_cols if c != 'agr_id_key']
    drop_before_merge = []
    for c in merge_payload_cols:
        for c_try in [c, f'{c}_x', f'{c}_y']:
            if c_try in base_df.columns:
                drop_before_merge.append(c_try)
    if drop_before_merge:
        base_df = base_df.drop(columns=sorted(set(drop_before_merge)))

    base_df = base_df.merge(
        actual_tariff_df[required_tariff_cols],
        on='agr_id_key',
        how='left'
    )

    base_df['tariff_name'] = base_df['tariff_name_actual'].where(
        base_df['tariff_name_actual'].notna() & (base_df['tariff_name_actual'].astype(str).str.strip() != ''),
        base_df['tariff_name_legacy']
    )
    base_df['tariff_source'] = base_df['tariff_name_actual'].apply(
        lambda x: 'agr_id_actual' if pd.notna(x) and str(x).strip() else 'legacy_cft'
    )
    base_df['commission_monthly'] = pd.to_numeric(base_df.get('commission_monthly_actual'), errors='coerce').where(
        pd.to_numeric(base_df.get('commission_monthly_actual'), errors='coerce').notna(),
        pd.to_numeric(base_df.get('commission_monthly_legacy'), errors='coerce')
    )

    commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
    commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
    int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
    amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)
    retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)

    base_df['commission_total'] = commission_from_ops_num + commission_monthly_num
    base_df['aur'] = retl_cnt_num * 1926
    base_df['chod'] = base_df['commission_total'] + int_component_num
    base_df['fin_result'] = base_df['chod'] - pd.to_numeric(base_df['aur'], errors='coerce').fillna(0) - amortization_num
    base_df['report_month'] = report_month_label
    base_df['snapshot_month_start'] = snapshot_month_start

    mvp_columns = [
        'report_month', 'snapshot_month_start', 'inn', 'company_name',
        'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
        'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
        'tariff_name', 'tariff_source',
        'retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
        'aur', 'amortization', 'chod', 'fin_result'
    ]

    for col in mvp_columns:
        if col not in base_df.columns:
            base_df[col] = None

    for col in [
        'trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
        'commission_total', 'aur', 'amortization', 'chod', 'fin_result'
    ]:
        base_df[col] = base_df[col].map(to_decimal_or_none)

    for col in ['retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt']:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

    final_df = base_df[mvp_columns].copy()

    if final_df is None:
        raise RuntimeError(f'[{report_month_label}] final_df is None')

    total_elapsed_sec = round(time.perf_counter() - month_total_start_ts, 2)
    _log_progress(f"final_df ready: rows={len(final_df):,}, total_elapsed={total_elapsed_sec}s")
    return final_df

In [ ]:
METRICS_ORDER = [
    'unique_inn',
    'retl_cnt',
    'term_cnt',
    'trx_cnt',
    'trx_sum',
    'commission_total',
    'int_component',
    'commission_monthly',
    'chod',
]


def build_lake_agg(final_df):
    if final_df is None or final_df.empty:
        return pd.DataFrame(
            columns=[
                'inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake', 'trx_cnt_lake',
                'trx_sum_lake', 'commission_total_lake', 'int_component_lake',
                'commission_monthly_lake', 'chod_lake'
            ]
        )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
    lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')

    lake_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_lake': 'max',
              'term_cnt_lake': 'max',
              'trx_cnt_lake': 'max',
              'trx_sum_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'commission_monthly_lake': 'max',
              'chod_lake': 'max',
          })
    )
    return lake_agg


def load_excel_agg(excel_path, excel_header=0):
    ex = pd.read_excel(excel_path, header=excel_header)

    col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_total_col': ['Комиссия эквайринга'],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'chod_col': ['ЧОД'],
    }

    resolved = {k: pick_col_robust(ex.columns, v) for k, v in col_map.items()}
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        raise ValueError(f'Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)

    ex['retl_cnt_excel'] = pd.to_numeric(ex[resolved['retl_col']], errors='coerce')
    ex['term_cnt_excel'] = pd.to_numeric(ex[resolved['term_col']], errors='coerce')
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved['trx_cnt_col']], errors='coerce')
    ex['trx_sum_excel'] = to_num_series(ex[resolved['trx_sum_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved['int_component_col']])
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['chod_excel'] = to_num_series(ex[resolved['chod_col']])

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_excel': 'max',
              'term_cnt_excel': 'max',
              'trx_cnt_excel': 'max',
              'trx_sum_excel': 'sum',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
              'commission_monthly_excel': 'max',
              'chod_excel': 'sum',
          })
    )
    return ex_agg, resolved


def calc_lake_totals(lake_agg):
    if lake_agg is None or lake_agg.empty:
        return {m: 0.0 if m != 'unique_inn' else 0 for m in METRICS_ORDER}

    return {
        'unique_inn': int(lake_agg['inn_key'].nunique()),
        'retl_cnt': float(lake_agg['retl_cnt_lake'].fillna(0).sum()),
        'term_cnt': float(lake_agg['term_cnt_lake'].fillna(0).sum()),
        'trx_cnt': float(lake_agg['trx_cnt_lake'].fillna(0).sum()),
        'trx_sum': float(lake_agg['trx_sum_lake'].fillna(0).sum()),
        'commission_total': float(lake_agg['commission_total_lake'].fillna(0).sum()),
        'int_component': float(lake_agg['int_component_lake'].fillna(0).sum()),
        'commission_monthly': float(lake_agg['commission_monthly_lake'].fillna(0).sum()),
        'chod': float(lake_agg['chod_lake'].fillna(0).sum()),
    }


def calc_excel_totals(ex_agg):
    if ex_agg is None or ex_agg.empty:
        return {m: 0.0 if m != 'unique_inn' else 0 for m in METRICS_ORDER}

    return {
        'unique_inn': int(ex_agg['inn_key'].nunique()),
        'retl_cnt': float(ex_agg['retl_cnt_excel'].fillna(0).sum()),
        'term_cnt': float(ex_agg['term_cnt_excel'].fillna(0).sum()),
        'trx_cnt': float(ex_agg['trx_cnt_excel'].fillna(0).sum()),
        'trx_sum': float(ex_agg['trx_sum_excel'].fillna(0).sum()),
        'commission_total': float(ex_agg['commission_total_excel'].fillna(0).sum()),
        'int_component': float(ex_agg['int_component_excel'].fillna(0).sum()),
        'commission_monthly': float(ex_agg['commission_monthly_excel'].fillna(0).sum()),
        'chod': float(ex_agg['chod_excel'].fillna(0).sum()),
    }


def build_monthly_compare(report_month_label, final_df, excel_path=None, excel_header=0):
    lake_agg = build_lake_agg(final_df)
    lake_totals = calc_lake_totals(lake_agg)

    if excel_path is None:
        rows = []
        for metric in METRICS_ORDER:
            rows.append({
                'report_month': report_month_label,
                'metric': metric,
                'lake_value': lake_totals[metric],
                'excel_value': np.nan,
                'delta_lake_minus_excel': np.nan,
                'divergence_pct': np.nan,
                'reference_status': 'no_excel_reference',
                'excel_path': None,
            })
        month_df = pd.DataFrame(rows)
        return month_df, lake_agg, None

    if not Path(excel_path).exists():
        raise FileNotFoundError(f'Excel reference not found for {report_month_label}: {excel_path}')

    ex_agg, resolved = load_excel_agg(excel_path, excel_header=excel_header)
    excel_totals = calc_excel_totals(ex_agg)

    rows = []
    for metric in METRICS_ORDER:
        lake_value = lake_totals[metric]
        excel_value = excel_totals[metric]
        delta = lake_value - excel_value
        rows.append({
            'report_month': report_month_label,
            'metric': metric,
            'lake_value': lake_value,
            'excel_value': excel_value,
            'delta_lake_minus_excel': delta,
            'divergence_pct': safe_divergence_pct(delta, excel_value),
            'reference_status': 'has_excel_reference',
            'excel_path': excel_path,
        })

    month_df = pd.DataFrame(rows)
    return month_df, lake_agg, {'excel_agg': ex_agg, 'resolved_columns': resolved}

In [ ]:
final_df_by_month = {}
monthly_compare_parts = []
lake_agg_by_month = {}
excel_meta_by_month = {}

for month_ts in period_months:
    report_month = month_ts.strftime('%Y-%m-%d')
    report_month_label = month_ts.strftime('%Y-%m')
    excel_path = excel_reference_by_month.get(report_month_label)

    month_start_ts = time.perf_counter()

    final_df_month = compute_final_df_for_month(report_month=report_month, imp=imp)
    final_df_by_month[report_month_label] = final_df_month

    month_compare_df, lake_agg_df, excel_meta = build_monthly_compare(
        report_month_label=report_month_label,
        final_df=final_df_month,
        excel_path=excel_path,
        excel_header=excel_header,
    )

    monthly_compare_parts.append(month_compare_df)
    lake_agg_by_month[report_month_label] = lake_agg_df
    excel_meta_by_month[report_month_label] = excel_meta

    elapsed = round(time.perf_counter() - month_start_ts, 2)
    print(
        f'[{report_month_label}] done: final_df_rows={len(final_df_month):,}, '
        f'ref_status={month_compare_df["reference_status"].iloc[0]}, elapsed_sec={elapsed}'
    )

if final_df_by_month:
    final_df_period_df = pd.concat(final_df_by_month.values(), ignore_index=True)
else:
    final_df_period_df = pd.DataFrame()

if monthly_compare_parts:
    monthly_compare_df = pd.concat(monthly_compare_parts, ignore_index=True)
else:
    monthly_compare_df = pd.DataFrame(
        columns=[
            'report_month', 'metric', 'lake_value', 'excel_value',
            'delta_lake_minus_excel', 'divergence_pct', 'reference_status', 'excel_path'
        ]
    )

print(f'final_df_period_df rows: {len(final_df_period_df):,}')
print(f'monthly_compare_df rows: {len(monthly_compare_df):,}')
display(monthly_compare_df.head(18))

In [ ]:
# Формируем period_total и reference_coverage
expected_months = [d.strftime('%Y-%m') for d in period_months]

reference_coverage_rows = []
for month_label in expected_months:
    excel_path = excel_reference_by_month.get(month_label)
    has_ref = excel_path is not None
    reference_coverage_rows.append({
        'report_month': month_label,
        'has_excel_reference': has_ref,
        'reference_status': 'has_excel_reference' if has_ref else 'no_excel_reference',
        'excel_path': excel_path,
    })

reference_coverage_df = pd.DataFrame(reference_coverage_rows)

jan_apr_compare = (
    monthly_compare_df[monthly_compare_df['reference_status'] == 'has_excel_reference']
    .groupby('metric', as_index=False)
    .agg({'lake_value': 'sum', 'excel_value': 'sum'})
)
jan_apr_compare['delta_lake_minus_excel'] = jan_apr_compare['lake_value'] - jan_apr_compare['excel_value']
jan_apr_compare['divergence_pct'] = jan_apr_compare.apply(
    lambda r: safe_divergence_pct(r['delta_lake_minus_excel'], r['excel_value']), axis=1
)
jan_apr_compare['scope'] = 'jan_apr_compare'
jan_apr_compare['reference_status'] = 'has_excel_reference'

may_jun_lake_only = (
    monthly_compare_df[monthly_compare_df['reference_status'] == 'no_excel_reference']
    .groupby('metric', as_index=False)
    .agg({'lake_value': 'sum'})
)
may_jun_lake_only['excel_value'] = np.nan
may_jun_lake_only['delta_lake_minus_excel'] = np.nan
may_jun_lake_only['divergence_pct'] = np.nan
may_jun_lake_only['scope'] = 'may_jun_lake_only'
may_jun_lake_only['reference_status'] = 'no_excel_reference'

period_total_df = pd.concat([jan_apr_compare, may_jun_lake_only], ignore_index=True)
period_total_df = period_total_df[
    ['scope', 'metric', 'lake_value', 'excel_value', 'delta_lake_minus_excel', 'divergence_pct', 'reference_status']
]

# Контроль по плану
if final_df_period_df.empty:
    raise RuntimeError('QC fail: final_df_period_df пустой')

final_months_present = sorted(final_df_period_df['report_month'].dropna().astype(str).unique().tolist())
if final_months_present != expected_months:
    raise RuntimeError(f'QC fail: months in final_df_period_df={final_months_present}, expected={expected_months}')

jan_apr = expected_months[:4]
may_jun = expected_months[4:]

jan_apr_cmp = monthly_compare_df[monthly_compare_df['report_month'].isin(jan_apr)]
for m in jan_apr:
    metrics_found = sorted(jan_apr_cmp[jan_apr_cmp['report_month'] == m]['metric'].tolist())
    if sorted(metrics_found) != sorted(METRICS_ORDER):
        raise RuntimeError(f'QC fail: month {m} has incomplete metric comparison: {metrics_found}')

may_jun_cmp = monthly_compare_df[monthly_compare_df['report_month'].isin(may_jun)]
if not may_jun_cmp.empty and may_jun_cmp['reference_status'].nunique() != 1:
    raise RuntimeError('QC fail: May-Jun have mixed reference_status values')
if not may_jun_cmp.empty and may_jun_cmp['reference_status'].iloc[0] != 'no_excel_reference':
    raise RuntimeError('QC fail: May-Jun are not marked as no_excel_reference')

# Экспорт
output_dir.mkdir(parents=True, exist_ok=True)

final_df_period_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

with pd.ExcelWriter(output_compare_excel_path, engine='openpyxl') as writer:
    monthly_compare_df.to_excel(writer, sheet_name='monthly_compare', index=False)
    period_total_df.to_excel(writer, sheet_name='period_total', index=False)
    reference_coverage_df.to_excel(writer, sheet_name='reference_coverage', index=False)

period_unique_inn = final_df_period_df['inn'].apply(normalize_inn_q1).dropna().nunique()

print('Saved files:')
print(f'  CSV:   {output_csv_path}')
print(f'  Excel: {output_compare_excel_path}')
print('Control stats:')
print(f'  final_df_period_df rows: {len(final_df_period_df):,}')
print(f'  final_df_period_df unique INN: {period_unique_inn:,}')
print(f'  months in final_df_period_df: {final_months_present}')

display(reference_coverage_df)
display(period_total_df)

In [ ]:
# DRP publish config (new table for Superset)
drp_schema_new = 'sbx_da'
drp_table_old = 'tmp_shestopalov_acq_datamart_q1'
drp_table_new = 'tmp_shestopalov_acq_datamart_q1_v2'

dashboard_required_cols = [
    'inn', 'company_name', 'contract_number',
    'd_valid_from', 'd_valid_to',
    'vsp_name', 'vsp_code',
    'retl_cnt', 'term_cnt', 'active_terms',
    'commission_total', 'chod', 'fin_result',
    'snapshot_month_start',
]

superset_time_col = 'snapshot_month_start'

print('DRP old table:', f"{drp_schema_new}.{drp_table_old}")
print('DRP new table:', f"{drp_schema_new}.{drp_table_new}")
print('Superset time column:', superset_time_col)
print('Required cols for smoke:', dashboard_required_cols)

In [ ]:
# Upload final_df_period_df to NEW DRP table + schema compatibility checks
if 'final_df_period_df' not in globals() or final_df_period_df is None or len(final_df_period_df) == 0:
    raise RuntimeError('Сначала выполните расчет периода: final_df_period_df пустой.')

schema_name = str(drp_schema_new).strip().lower()
old_table_name = str(drp_table_old).strip().lower()
new_table_name = str(drp_table_new).strip().lower()

target_old_fq = f'{schema_name}.{old_table_name}'
target_new_fq = f'{schema_name}.{new_table_name}'

print('Preparing upload...')
print('old table =', target_old_fq)
print('new table =', target_new_fq)
print('source rows =', len(final_df_period_df))

drp_user = input('DRP user: ').strip()
drp_password = getpass('DRP password: ')

drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': drp_user,
        'password': drp_password,
    }
)

df_upload = final_df_period_df.copy()
df_upload.columns = [str(c).strip().lower() for c in df_upload.columns]

sql_old_cols = f"""
select lower(column_name) as column_name
from information_schema.columns
where table_schema = '{schema_name}'
  and table_name = '{old_table_name}'
order by ordinal_position
"""

with drp_conn:
    old_cols_df = drp_conn.fetch(sql_old_cols)

old_cols = []
if old_cols_df is not None and len(old_cols_df):
    old_cols = [str(x).strip().lower() for x in old_cols_df['column_name'].dropna().tolist()]

print('old table columns =', len(old_cols))

# Compatibility aliases for existing dashboard fields
compat_alias_map = {
    'branch_rf': 'filial_rf',
    'tarif_name': 'tariff_name',
    'month': 'snapshot_month_start',
}
for target_col, source_col in compat_alias_map.items():
    if target_col in old_cols and target_col not in df_upload.columns and source_col in df_upload.columns:
        df_upload[target_col] = df_upload[source_col]

missing_old_cols_before_fill = [c for c in old_cols if c not in df_upload.columns]
for c in missing_old_cols_before_fill:
    df_upload[c] = None

if old_cols:
    extra_new_cols = [c for c in df_upload.columns if c not in old_cols]
    df_upload = df_upload[old_cols + extra_new_cols]

# Prepare DRP upload as TEXT columns
upload_df = df_upload.copy()
for c in upload_df.columns:
    upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x))
    upload_df[c] = upload_df[c].astype(object)

col_defs = []
for c in upload_df.columns:
    col_name = str(c).replace('"', '""')
    col_defs.append(f'"{col_name}" TEXT')

create_sql = f"""
CREATE TABLE {target_new_fq}
(
  {', '.join(col_defs)}
)
"""

with drp_conn:
    drp_conn.execute(f'DROP TABLE IF EXISTS {target_new_fq}')
    drp_conn.execute(create_sql)
    drp_conn.write(
        table=target_new_fq,
        df=upload_df,
        mode='append',
    )

    cnt_new_df = drp_conn.fetch(f"select count(*) as row_cnt from {target_new_fq}")
    new_cols_df = drp_conn.fetch(f"""
        select lower(column_name) as column_name
        from information_schema.columns
        where table_schema = '{schema_name}'
          and table_name = '{new_table_name}'
        order by ordinal_position
    """)

new_cols = [str(x).strip().lower() for x in new_cols_df['column_name'].dropna().tolist()] if new_cols_df is not None and len(new_cols_df) else []
missing_old_in_new = [c for c in old_cols if c not in new_cols]
required_missing = [c for c in dashboard_required_cols if c not in new_cols]

new_row_cnt = int(pd.to_numeric(cnt_new_df.iloc[0, 0], errors='coerce')) if cnt_new_df is not None and len(cnt_new_df) else 0

# Basic KPI sanity (from uploaded dataframe before string cast)
def _safe_sum(df, col):
    if col not in df.columns:
        return None
    return float(pd.to_numeric(df[col], errors='coerce').fillna(0).sum())

kpi_sanity = pd.DataFrame([
    {'metric': 'row_cnt_new_table', 'value': new_row_cnt},
    {'metric': 'sum_retl_cnt', 'value': _safe_sum(df_upload, 'retl_cnt')},
    {'metric': 'sum_term_cnt', 'value': _safe_sum(df_upload, 'term_cnt')},
    {'metric': 'sum_active_terms', 'value': _safe_sum(df_upload, 'active_terms')},
    {'metric': 'sum_commission_total', 'value': _safe_sum(df_upload, 'commission_total')},
    {'metric': 'sum_chod', 'value': _safe_sum(df_upload, 'chod')},
    {'metric': 'sum_fin_result', 'value': _safe_sum(df_upload, 'fin_result')},
])

compat_report = pd.DataFrame([
    {'check': 'old_columns_count', 'value': len(old_cols)},
    {'check': 'new_columns_count', 'value': len(new_cols)},
    {'check': 'missing_old_in_new_count', 'value': len(missing_old_in_new)},
    {'check': 'required_missing_count', 'value': len(required_missing)},
])

print('Upload finished:', target_new_fq)
print('Missing old columns in new:', missing_old_in_new[:20])
print('Missing required dashboard columns:', required_missing)

display(compat_report)
display(kpi_sanity)

display(pd.DataFrame({'missing_old_in_new': missing_old_in_new}))
display(pd.DataFrame({'missing_required': required_missing}))

## Superset switch checklist (new table)

1) Create new Dataset on table `sbx_da.tmp_shestopalov_acq_datamart_q1_v2`.
2) Set temporal column to `snapshot_month_start`.
3) Rebind charts in a dashboard copy first, then publish.
4) Update KPI **Количество активных терминалов** metric to:

```sql
SUM(active_terms)
```

5) Smoke-check after switch:
- all filters work (`Региональный филиал`, `Наименование клиента`, `ИНН клиента`, `Тариф`, `Сегмент`, `Month`);
- KPI cards render without SQL errors;
- chart `ФР и ЧОД` renders for selected months;
- client table renders and sorts.

Rollback:
- keep old dataset/table untouched;
- if issue occurs, point charts back to dataset on `sbx_da.tmp_shestopalov_acq_datamart_q1`.